# AIC2026 — TARA text encoder server (Colab A100)

Serve `TARA.encode_text(...)` in the **same checkpoint and embedding space** as `tara-tarsier2-7b-3584-clip-v1`: commit `3e7cb730d86ae10da7eb1c85f17e45ece5a5e353`, fingerprint `ef197331649dfb93e7057304f294d8bd1cd98bf22b1902bca48daf6dc4874819`, FP32 L2-normalized 3584-d. Text queries should be translated to English before calling this server.

Run cells from top to bottom on a Colab **A100 GPU**. The server uses a pinned Python 3.10 venv because the original TARA video notebook uses Torch 2.5.1+cu121, Transformers 4.45.0 and FlashAttention 2.8.3. The model stays loaded between requests. Keep the last cell and Colab runtime alive while searching.

Colab Secrets: `HF_TOKEN` if the checkpoint requires authentication, `TARA_ENCODER_TOKEN` optionally for a stable bearer token. For a stable public URL use a **dedicated** Cloudflare tunnel secret `CF_TUNNEL_TOKEN_TARA` and `TARA_ENCODER_HOSTNAME`; otherwise the notebook starts a quick tunnel.

In [ ]:
from pathlib import Path
import hashlib, os, platform, secrets, shutil, subprocess, sys, time
import requests

try:
    from google.colab import userdata
    def secret(name):
        try:
            value = userdata.get(name)
            return value.strip() if value and value.strip() else None
        except Exception:
            return None
except ImportError:
    def secret(name): return None

if subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    raise RuntimeError('Select an A100 GPU runtime in Colab.')
GPU_NAME = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True).strip().splitlines()[0]
if 'A100' not in GPU_NAME.upper():
    raise RuntimeError(f'This notebook expects A100, got {GPU_NAME}')
HF_TOKEN = secret('HF_TOKEN') or os.environ.get('HF_TOKEN') or ''
SERVICE_TOKEN = secret('TARA_ENCODER_TOKEN') or os.environ.get('TARA_ENCODER_TOKEN') or secrets.token_urlsafe(32)
if len(SERVICE_TOKEN) < 24:
    raise RuntimeError('TARA_ENCODER_TOKEN must be at least 24 characters')
VENV = Path('/content/tara-text-venv')
PYTHON = VENV / 'bin/python'
print('GPU:', GPU_NAME, '| kernel:', platform.python_version(), '| bearer token configured:', bool(SERVICE_TOKEN))

## 1. Pinned TARA environment

The FA2 wheel is the same official binary and SHA-256 used by the video-embedding notebook. This cell can be rerun; it skips an already verified venv.

In [ ]:
import urllib.request

def run(*cmd):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

marker = VENV / '.tara-text-contract'
contract = 'python3.10-torch2.5.1cu121-transformers4.45.0-fa2.8.3-tara-text-v1'
if not PYTHON.is_file() or not marker.is_file() or marker.read_text() != contract:
    run(sys.executable, '-m', 'pip', 'install', '-q', 'uv')
    uv = shutil.which('uv') or str(Path.home() / '.local/bin/uv')
    run(uv, 'python', 'install', '3.10')
    if VENV.exists(): shutil.rmtree(VENV)
    run(uv, 'venv', '--python', '3.10', VENV)
    run(uv, 'pip', 'install', '--python', PYTHON, '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.5.1', 'torchvision==0.20.1')
    packages = [
        'transformers==4.45.0','tokenizers==0.20.3','accelerate==0.34.2',
        'huggingface-hub==0.35.3','safetensors==0.4.5','sentencepiece==0.2.1',
        'numpy==1.26.4','pandas==2.2.3','pyarrow==17.0.0','Pillow==10.0.0',
        'decord==0.6.0','einops==0.8.1','opencv-python-headless==4.11.0.86',
        'matplotlib==3.9.2','seaborn==0.13.2','scikit-learn==1.5.2',
        'ipython==8.27.0','ipywidgets==8.1.5','ffmpeg-python==0.2.0',
        'func-timeout==4.3.5','termcolor==2.4.0','PyYAML==6.0.2',
        'tqdm==4.67.1','psutil==6.1.0','fastapi==0.115.6',
        'pydantic==2.10.6','uvicorn==0.34.0',
    ]
    run(uv, 'pip', 'install', '--python', PYTHON, *packages)
    wheel_url = 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl'
    wheel = Path('/content') / wheel_url.rsplit('/', 1)[-1]
    if not wheel.is_file() or wheel.stat().st_size != 255_955_166 or sha256_file(wheel) != 'c59be18fa934e132e5a405bcca6673af1d9d09a0036a9e081133bc7b7fc2992e':
        temp = wheel.with_suffix('.partial')
        urllib.request.urlretrieve(wheel_url, temp)
        if temp.stat().st_size != 255_955_166 or sha256_file(temp) != 'c59be18fa934e132e5a405bcca6673af1d9d09a0036a9e081133bc7b7fc2992e':
            raise RuntimeError('FlashAttention wheel hash mismatch')
        temp.replace(wheel)
    run(uv, 'pip', 'install', '--python', PYTHON, '--no-deps', wheel)
    marker.write_text(contract)
probe = subprocess.check_output([str(PYTHON), '-c', 'import torch,transformers,flash_attn,numpy; print(torch.__version__,transformers.__version__,flash_attn.__version__,numpy.__version__,torch.cuda.is_available())'], text=True).strip()
if not ('2.5.1' in probe and '4.45.0' in probe and '2.8.3' in probe and '1.26.4' in probe and 'True' in probe):
    raise RuntimeError('TARA venv failed its version/CUDA probe: ' + probe)
print('TARA venv PASS:', probe)

## 2. Start the authenticated text worker

The embedded worker verifies the model checkpoint files, loads the official `TARA.from_pretrained`, and checks the runtime contract before serving. `/health` exposes revision/fingerprint/dimension; `/encode-text` requires a bearer token.

In [ ]:
from pathlib import Path
SERVER_SOURCE = '#!/usr/bin/env python3\n"""TARA text-only FastAPI worker for the AIC2026 Colab notebook.\n\nThis file is embedded verbatim in the notebook so the notebook is standalone.\nRun under its pinned Python 3.10 environment, with TARA_SERVICE_TOKEN set.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport hmac\nimport json\nimport os\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\nfrom typing import Annotated\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom fastapi import FastAPI, Header, HTTPException\nfrom huggingface_hub import snapshot_download\nfrom pydantic import BaseModel, ConfigDict, Field, StringConstraints, model_validator\n\nMODEL_ID = "bpiyush/TARA"\nMODEL_REVISION = "3e7cb730d86ae10da7eb1c85f17e45ece5a5e353"\nSEMANTIC_FINGERPRINT = "ef197331649dfb93e7057304f294d8bd1cd98bf22b1902bca48daf6dc4874819"\nDIM = 3584\nMAX_TEXTS = 16\nMAX_CHARS = 4096\nMAX_TOTAL_CHARS = 16_384\nSERVICE_TOKEN = os.environ.get("TARA_SERVICE_TOKEN", "")\nHF_TOKEN = os.environ.get("HF_TOKEN") or None\nif len(SERVICE_TOKEN) < 24:\n    raise RuntimeError("TARA_SERVICE_TOKEN must have at least 24 characters")\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA GPU is required")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        while block := handle.read(8 << 20):\n            digest.update(block)\n    return digest.hexdigest()\n\n\nsnapshot = Path(snapshot_download(\n    repo_id=MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN,\n    cache_dir="/content/tara-model-cache", ignore_patterns=["*.png", "*.md", "assets/*"],\n))\nchecks = {\n    "model.safetensors.index.json": "59a64242446007de3eb6502c7d2ab52cc2721a0037fb920ae716b9218a4ad8af",\n    "modeling_tara.py": "793b51bd4df2427e62c70737ccab4a3dbe2ca3173e5f762f6b2e8a5b1b78a9b0",\n    "tarsier2/default_config.yaml": "d1708c94328c66c518db434ae88939cb071d9f03e9cabb47eac27ea7fae35443",\n}\nfor name, expected in checks.items():\n    if sha256_file(snapshot / name) != expected:\n        raise RuntimeError(f"Checkpoint file hash mismatch: {name}")\nindex = json.loads((snapshot / "model.safetensors.index.json").read_text())\nif int(index.get("metadata", {}).get("total_size", -1)) != 16_582_751_232:\n    raise RuntimeError("Checkpoint tensor size differs from the video artifact")\nfor name in set(index["weight_map"].values()):\n    if not (snapshot / name).is_file():\n        raise RuntimeError(f"Missing weight shard: {name}")\n\nsys.path.insert(0, str(snapshot))\ntry:\n    import shared.utils  # noqa: F401\nexcept Exception:\n    # Same minimal fallback used by the video-embedding notebook. The official\n    # TARA loader only needs YAML loading and its repository root here.\n    import types\n    import yaml\n\n    shared = sys.modules.get("shared") or types.ModuleType("shared")\n    utils = types.ModuleType("shared.utils")\n    io = types.ModuleType("shared.utils.io")\n    log = types.ModuleType("shared.utils.log")\n    io.load_yml = lambda path: yaml.safe_load(Path(path).read_text())\n    log.repo_path = str(snapshot)\n    utils.io = io\n    utils.log = log\n    shared.utils = utils\n    sys.modules.update({\n        "shared": shared, "shared.utils": utils,\n        "shared.utils.io": io, "shared.utils.log": log,\n    })\n\nfrom modeling_tara import TARA  # noqa: E402\n\nMODEL = TARA.from_pretrained(\n    str(snapshot), device_map={"": 0},\n    attn_implementation="flash_attention_2", low_cpu_mem_usage=True,\n)\nMODEL.model.requires_grad_(False)\nMODEL.model.eval()\nPARAMETER = next(MODEL.model.parameters())\nif PARAMETER.device.type != "cuda" or PARAMETER.dtype != torch.bfloat16:\n    raise RuntimeError("TARA is not resident on CUDA in bfloat16")\nif MODEL.processor.tokenizer.padding_side != "left":\n    raise RuntimeError("TARA text pooling requires left padding")\nif int(MODEL.model.config.text_config.hidden_size) != DIM:\n    raise RuntimeError("TARA hidden size differs from video embeddings")\nif MODEL.video_eol_prompt != "<video>\\nSummary above video in one word:":\n    raise RuntimeError("Unexpected TARA video prompt")\n\nLOCK = threading.Lock()\nText = Annotated[str, StringConstraints(strip_whitespace=True, min_length=1, max_length=MAX_CHARS)]\n\n\nclass EncodeRequest(BaseModel):\n    model_config = ConfigDict(extra="forbid")\n    texts: list[Text] = Field(min_length=1, max_length=MAX_TEXTS)\n\n    @model_validator(mode="after")\n    def total_chars(self):\n        if sum(map(len, self.texts)) > MAX_TOTAL_CHARS:\n            raise ValueError("Total text length exceeds limit")\n        return self\n\n\ndef require_bearer(authorization: str | None) -> None:\n    scheme, _, token = (authorization or "").partition(" ")\n    if scheme.lower() != "bearer" or not hmac.compare_digest(token, SERVICE_TOKEN):\n        raise HTTPException(status_code=401, detail="Invalid bearer token")\n\n\n@torch.inference_mode()\ndef encode_text(texts: list[str]) -> np.ndarray:\n    with LOCK:\n        raw = MODEL.encode_text(texts)\n        vectors = F.normalize(raw.float(), p=2, dim=-1).cpu().numpy().astype(np.float32)\n    if vectors.shape != (len(texts), DIM) or not np.isfinite(vectors).all():\n        raise RuntimeError("TARA returned wrong shape or nonfinite vectors")\n    if not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=2e-5, rtol=0):\n        raise RuntimeError("TARA vectors are not unit norm")\n    return vectors\n\n\nAPP = FastAPI(title="AIC2026 TARA Text Encoder", version="1.0.0")\n\n\n@APP.get("/health")\ndef health():\n    return {\n        "ok": True, "model": MODEL_ID, "revision": MODEL_REVISION,\n        "semantic_fingerprint": SEMANTIC_FINGERPRINT,\n        "dim": DIM, "l2_normalized": True,\n        "device": str(PARAMETER.device), "dtype": str(PARAMETER.dtype),\n        "gpu": torch.cuda.get_device_name(0),\n        "max_texts": MAX_TEXTS, "max_chars": MAX_CHARS,\n    }\n\n\n@APP.post("/encode-text")\ndef api_encode_text(req: EncodeRequest, authorization: str | None = Header(default=None)):\n    require_bearer(authorization)\n    started = time.perf_counter()\n    try:\n        vectors = encode_text(req.texts)\n    except torch.OutOfMemoryError as exc:\n        torch.cuda.empty_cache()\n        raise HTTPException(status_code=503, detail="GPU out of memory") from exc\n    return {\n        **health(), "latency_ms": round((time.perf_counter() - started) * 1000, 2),\n        "vectors": vectors.tolist(),\n    }\n\n\nif __name__ == "__main__":\n    import uvicorn\n\n    uvicorn.run(APP, host="0.0.0.0", port=int(os.environ.get("TARA_PORT", "8001")))\n'
SERVER_FILE = Path("/content/tara_text_server.py")
SERVER_FILE.write_text(SERVER_SOURCE, encoding="utf-8")
print("Wrote", SERVER_FILE, "bytes", SERVER_FILE.stat().st_size)

In [ ]:
import json, numpy as np
PORT = 8001
LOCAL_URL = f'http://127.0.0.1:{PORT}'
if 'SERVER' in globals() and SERVER.poll() is None:
    SERVER.terminate()
    SERVER.wait(timeout=20)
server_env = os.environ.copy()
server_env.update(HF_TOKEN=HF_TOKEN, TARA_SERVICE_TOKEN=SERVICE_TOKEN, TARA_PORT=str(PORT), TOKENIZERS_PARALLELISM='false')
LOG_FILE = Path('/content/tara-text-server.log')
LOG_HANDLE = LOG_FILE.open('w')
SERVER = subprocess.Popen([str(PYTHON), '-u', str(SERVER_FILE)], env=server_env, stdout=LOG_HANDLE, stderr=subprocess.STDOUT)
deadline = time.monotonic() + 1800
while time.monotonic() < deadline:
    if SERVER.poll() is not None:
        raise RuntimeError('TARA worker stopped during startup:\n' + '\n'.join(LOG_FILE.read_text(errors='replace').splitlines()[-30:]))
    try:
        health = requests.get(LOCAL_URL + '/health', timeout=3).json()
        if health.get('ok'): break
    except Exception: pass
    time.sleep(5)
else:
    raise RuntimeError('TARA worker did not become healthy in 30 minutes. See ' + str(LOG_FILE))
assert health['model'] == 'bpiyush/TARA'
assert health['revision'] == '3e7cb730d86ae10da7eb1c85f17e45ece5a5e353'
assert health['semantic_fingerprint'] == 'ef197331649dfb93e7057304f294d8bd1cd98bf22b1902bca48daf6dc4874819'
assert health['dim'] == 3584 and health['l2_normalized'] is True
print('Health PASS:', health)
assert requests.post(LOCAL_URL + '/encode-text', json={'texts':['a chef cuts shrimp']}, timeout=10).status_code == 401
response = requests.post(LOCAL_URL + '/encode-text', json={'texts':['a chef cuts shrimp, then places it on a plate','a cyclist races uphill']}, headers={'Authorization':'Bearer '+SERVICE_TOKEN}, timeout=180)
response.raise_for_status()
wire = response.json()
vectors = np.asarray(wire['vectors'], dtype=np.float32)
assert vectors.shape == (2,3584) and np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=2e-5, rtol=0)
print('Local authenticated encode PASS:', vectors.shape, 'latency_ms=', wire['latency_ms'])

## 3. Cloudflare tunnel and remote smoke test

Use a dedicated named tunnel for TARA, routed to `http://localhost:8001`, or let this cell create a quick tunnel. The last cell prints backend environment values only after checking the public endpoint.

In [ ]:
import re, threading
from collections import deque
CF_TOKEN = secret('CF_TUNNEL_TOKEN_TARA') or os.environ.get('CF_TUNNEL_TOKEN_TARA') or ''
HOSTNAME = (secret('TARA_ENCODER_HOSTNAME') or os.environ.get('TARA_ENCODER_HOSTNAME') or '').strip().removeprefix('https://').rstrip('/')
if CF_TOKEN and not HOSTNAME:
    raise RuntimeError('Set TARA_ENCODER_HOSTNAME for the dedicated named tunnel')
asset = {'x86_64':'cloudflared-linux-amd64','aarch64':'cloudflared-linux-arm64'}.get(platform.machine().lower())
if not asset: raise RuntimeError('Unsupported CPU architecture')
cloudflared = Path('/content/cloudflared')
if not cloudflared.is_file() or cloudflared.stat().st_size < 10_000_000:
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/'+asset, cloudflared)
cloudflared.chmod(0o755)
if 'TUNNEL' in globals() and TUNNEL.poll() is None:
    TUNNEL.terminate(); TUNNEL.wait(timeout=20)
command = [str(cloudflared),'tunnel','--no-autoupdate','--protocol','http2','--loglevel','info']
command += ['run','--token',CF_TOKEN] if CF_TOKEN else ['--url',LOCAL_URL]
TUNNEL = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = deque(maxlen=100)
def drain():
    for line in TUNNEL.stdout:
        safe = line.rstrip().replace(CF_TOKEN, '<redacted>') if CF_TOKEN else line.rstrip()
        lines.append(safe)
threading.Thread(target=drain, daemon=True).start()
PUBLIC_URL = 'https://' + HOSTNAME if CF_TOKEN else None
deadline = time.monotonic() + 120
while time.monotonic() < deadline:
    if TUNNEL.poll() is not None: raise RuntimeError('Tunnel exited: '+ '\n'.join(lines))
    if not CF_TOKEN:
        for line in lines:
            found = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
            if found: PUBLIC_URL = found.group(0); break
    if PUBLIC_URL:
        try:
            remote = requests.get(PUBLIC_URL+'/health', timeout=8)
            if remote.ok and remote.json().get('semantic_fingerprint') == health['semantic_fingerprint']: break
        except Exception: pass
    time.sleep(2)
else:
    raise RuntimeError('Public TARA tunnel health failed. Recent logs:\n'+'\n'.join(lines))
remote = requests.post(PUBLIC_URL+'/encode-text', json={'texts':['a chef cuts shrimp, then places it on a plate']}, headers={'Authorization':'Bearer '+SERVICE_TOKEN}, timeout=180)
remote.raise_for_status()
assert np.asarray(remote.json()['vectors'], dtype=np.float32).shape == (1,3584)
print('Public TARA encode PASS')
print('TARA_ENABLED=true')
print('TARA_ENCODER_URL='+PUBLIC_URL)
print('TARA_ENCODER_TOKEN='+SERVICE_TOKEN)
print('MILVUS_TARA_COLLECTION_2=aic26_tara_clips_infoshotpp_v1')

## Deployment notes

Keep this Colab runtime running. Set the four printed values in `backend/.env` after `milvus_upload_tara.py` reports 168,536 indexed rows. In InfoShot++, TARA searches event/sequence/scene separately and fuses video ranks with PE/Qwen. TRAKE searches event clips and maps each clip to a real keyframe before temporal DP. `TARA_ENABLED=false` restores the previous retrieval path without changing indexes.